# RIGHTWARE SIC — E1-X03 | Evidence Support & Answerability
**LEARN · Experimental · CPU · Offline · Run All**

## Goal
Investigate the X02 failure: property mentions are not asserted values. Separate scoped retrieval, support extraction, learned verification and abstention/conflict. New same-author synthetic data; no human gold, independent holdout, sealed A0, E1.1B eval, vendor or private input. Official `E1_DATA_READY=REVIEW_REQUIRED`.

Kaggle: import notebook, Accelerator None, Internet Off, Run All. Download `RIGHTWARE_SIC_E1_X03_EVIDENCE.zip` from Output. No additional files or installation.

## Context & methods
Source: [X02 audit SI-MH-0090](https://github.com/rightware-corporations/rightware-shared-intelligence/blob/17dbc2523b09cec494c7a72d96e5eebed34ba9b1/docs/research/SI_MH_0090_E1_X02_KAGGLE_AUDIT.md).

Fixed design before execution: 72 disjoint entity groups (40 train, 16 dev, 16 diagnostic), nine questions and eleven documents each. Two answerable cases, six negative/ambiguous cases and one conflict. Train-only vocabulary, statistics and model fitting. Scope-only BM25 is a separate control; all other ranking methods use the same eligible official/current/scope pool. Refit pointwise, pairwise and support verifier on new data; X02 is historical, not a matched cross-version experiment. Compare mention, assertion-rule and learned policies on same explicit top-5 shortlist. Correct acceptance requires exact source/property/value/span. Conflict signals evaluated separately. Threshold grid 0..1 plus 1.01, maximize correct dev accepts with zero wrong accepts, ties higher threshold. Diagnostic not used for tuning. Scores not calibrated probabilities. Rule labels and grammar authored together; no independent evaluation or production authorization.

In [ ]:
import os
os.environ['OMP_NUM_THREADS']='1';os.environ['OPENBLAS_NUM_THREADS']='1';os.environ['MKL_NUM_THREADS']='1'
import re,json,math,random,hashlib,zipfile,sys,platform,uuid
from pathlib import Path
from datetime import datetime,timezone
from collections import Counter,defaultdict
import numpy as np,pandas as pd,sklearn,matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from IPython.display import display
SEED=20261126;rng=random.Random(SEED);K=5
ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUT=ROOT/('SIC_E1_X03_'+uuid.uuid4().hex[:10]);OUT.mkdir()
ZIP=ROOT/'RIGHTWARE_SIC_E1_X03_EVIDENCE.zip'
CFG={'schema':'RIGHTWARE_E1_X03_V1','seed':SEED,'groups':{'train':40,'dev':16,'diagnostic':16},'k':K,'grid':[round(i/100,2) for i in range(101)]+[1.01],'max_wrong_dev':0,'science_gate':'REVIEW_REQUIRED','shortlist':'Explicit','classification':'PROVISIONAL_SYNTHETIC_NOT_INDEPENDENT','source':'SI-MH-0090'}
def write(name,obj): (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
def lines(name,rows): (OUT/name).write_text(''.join(json.dumps(r,ensure_ascii=False,allow_nan=False)+'\n' for r in rows),encoding='utf-8')
write('protocol.json',CFG);print('CPU',sys.version.split()[0],np.__version__,sklearn.__version__)

## Data
### 1. New corpus
Bilingual property aliases, new random entity/alias tokens, cross-mentions, wrong-entity distractors, negated and uncertain values, foreign/expired records and contradictory official values. Diagnostic introduces new sentence order and uncertainty phrasing. Oracle support spans are constructed from the generated clauses and kept separate. The labels are rules, not human judgments.

In [ ]:
A={'budget':[['orçamento','verba','dotação'],['budget','funding','allocation']], 'deadline':[['data limite','prazo'],['due date','deadline']], 'capacity':[['capacidade','limite','lotação'],['capacity','limit']], 'location':[['localização','local'],['location','site']], 'status':[['estado','situação'],['status','stage']], 'duration':[['duração','período'],['duration','period']], 'temperature':[['temperatura','temperatura alvo'],['temperature','target temperature']], 'pressure':[['pressão','pressão alvo'],['pressure','target pressure']]}
props=list(A);used=set();ASOF='2026-06-15'
def opaque(prefix,n=12):
    while True:
        v=prefix+''.join(rng.choices('0123456789ABCDEF',k=n))
        if v not in used:used.add(v);return v
def value(p):
    if p=='budget':return 'MT '+str(rng.randint(1200,95000))
    if p=='deadline':return '2026-08-'+str(rng.randint(10,28))
    if p=='capacity':return str(rng.randint(20,390))+' units'
    if p=='location':return 'ZONE-'+''.join(rng.choices('ABCDEFGH',k=3))
    if p=='status':return rng.choice(['APPROVED','IN_REVIEW','PENDING','SUSPENDED'])
    if p=='duration':return str(rng.randint(10,90))+' days'
    if p=='temperature':return str(rng.randint(15,85))+' °C'
    return str(rng.randint(11,90))+' kPa'
def term(p,l):return rng.choice(A[p][l])
def assertion(e,p,v,l,s):
    t=term(p,l)
    options={'train':[[f'Registo {e}: {t} = {v}',f'{t} de {e} é {v}'],[f'{e} has {t} of {v}',f'For {e}, {t}: {v}']], 'dev':[[f'Em {e}, {t} foi fixado em {v}',f'{t} de {e} consta como {v}'],[f'{t} for {e} stands at {v}',f'{e} records {t} as {v}']], 'diagnostic':[[f'{v} — valor que consta para {t} de {e}',f'{e} tem {t}: {v}',f'{t} de {e} permanece em {v}'],[f'{v} is recorded as the {t} for {e}',f'For {e}, {t} totals {v}',f'{e} records {t}: {v}']]}
    return rng.choice(options[s][l])
def nonassert(e,p,v,l,k,s):
    t=term(p,l)
    forms={'missing':[[f'O índice de {e} menciona {t}',f'Consultar {t} de {e} noutro dossier'],[f'{e} index mentions {t}',f'No value recorded for {t} of {e}']], 'negated':[[f'{e}: {t} = {v} foi rejeitado',f'{t} de {e} não é {v}'],[f'{e}: {t} = {v} was rejected',f'{t} of {e} is not {v}']], 'uncertain':[[f'Proposta para {e}: {t} = {v}',f'{e}: {t} = {v} por validar'],[f'Proposed {t} of {e}: {v}',f'{e}: {t} = {v} awaiting validation']]}
    if s=='diagnostic':forms['uncertain'][l].append(f'{e}: {t} = {v} (cenário de estudo)' if l==0 else f'{e}: {t} = {v} (planning scenario)')
    return rng.choice(forms[k][l])
def qtext(e,p,l,s):
    t=term(p,l);return rng.choice({'train':[[f'Qual é {t} de {e}?',f'Indique {t} para {e}.'],[f'What is {t} of {e}?',f'Find {t} for {e}.']], 'dev':[[f'Preciso de saber {t} relativo a {e}.'],[f'Retrieve the recorded {t} for {e}.']], 'diagnostic':[[f'No registo de {e}, que valor consta para {t}?'],[f'For {e}, which value is on record for {t}?']]}[s][l])
docs=[];queries=[];gold={};gi=0
for split,count in CFG['groups'].items():
 for _ in range(count):
    gi+=1;group=opaque('G-',10);entity=opaque('RW-',10);alias=opaque('NOVA-',8);both=entity+' ('+alias+')';scope='lab:'+str(gi%2)
    p1,p2,pm,pn,pu,pf,pe,pc=rng.sample(props,8);support={}
    def add(p,k,v=None):
        l=rng.randrange(2);v=v or value(p);sid=opaque('S-');cl=nonassert(both,p,v,l,k,split) if k in ('missing','negated','uncertain') else assertion(both,p,v,l,split)
        other=term(rng.choice([x for x in props if x!=p]),l);note=f'Nota administrativa sobre {other} de {both}' if l==0 else f'Administrative reference to {other} of {both}'
        if k=='wrong_entity':cl=assertion(opaque('RW-',10),p,v,l,split);note=f'Index linked to {both}'
        clauses=[cl,note];rng.shuffle(clauses);body='; '.join(clauses)+'.'
        docs.append({'source_id':sid,'split':split,'scope':'lab:foreign' if k=='foreign' else scope,'authority':'UNVERIFIED_CLAIM' if k=='unverified' else 'OFFICIAL_SOURCE','observed_at':'2026-01-10','valid_from':'2026-01-01','valid_until':'2026-04-01' if k=='expired' else None,'text':body,'language':['pt','en'][l]})
        return {'source_id':sid,'property':p,'value':v,'start':body.index(cl),'end':body.index(cl)+len(cl),'quote':cl}
    for p in (p1,p2):support[p]=[add(p,'positive')]
    for p,k in [(pm,'missing'),(pn,'negated'),(pu,'uncertain'),(pf,'foreign'),(pe,'expired')]:add(p,k)
    a=add(pc,'conflict');v=value(pc)
    while v==a['value']:v=value(pc)
    support[pc]=[a,add(pc,'conflict',v)];add(p1,'unverified');add(p2,'wrong_entity')
    for p,k in [(p1,'ANSWERABLE_ID'),(p2,'ANSWERABLE_ALIAS'),(pm,'MISSING'),(pn,'NEGATED'),(pu,'UNCERTAIN'),(pf,'FOREIGN_ONLY'),(pe,'EXPIRED_ONLY'),(pc,'CONFLICT'),(None,'AMBIGUOUS')]:
        l=rng.randrange(2);qid=opaque('Q-');ref=alias if k=='ANSWERABLE_ALIAS' else entity
        text=(f'Para {ref}, qual é {term(p1,l)} ou {term(p2,l)}?' if l==0 else f'For {ref}, what is {term(p1,l)} or {term(p2,l)}?') if p is None else qtext(ref,p,l,split)
        queries.append({'query_id':qid,'split':split,'scope':scope,'as_of':ASOF,'language':['pt','en'][l],'text':text});gold[qid]={'query_id':qid,'group':group,'class':k,'property':p,'support':support.get(p,[]) if k.startswith('ANSWERABLE') or k=='CONFLICT' else []}
rng.shuffle(docs);rng.shuffle(queries);D={d['source_id']:d for d in docs};Q={q['query_id']:q for q in queries}
assert len(D)==len(docs)==792 and len(Q)==len(queries)==648 and set(Q)==set(gold)
assert len({d['text'] for d in docs})==len(docs)
assert all(len({Q[qid]['split'] for qid,g in gold.items() if g['group']==gr})==1 for gr in {g['group'] for g in gold.values()})
assert all(D[a['source_id']]['text'][a['start']:a['end']]==a['quote'] for g in gold.values() for a in g['support'])
lines('documents.jsonl',docs);lines('queries.jsonl',queries);lines('rule_gold.jsonl',gold.values())
print(len(docs),'docs',len(queries),'queries');print(pd.crosstab([q['split'] for q in queries],[gold[q['query_id']]['class'] for q in queries]).to_string())

### 2. Literal evidence parser and features
Longest nonoverlapping aliases, typed value, same-clause entity and property, connector, negation/uncertainty cues. The parser has intentionally restricted grammar; a random alias token is not realistic entity resolution. No oracle labels in features.

In [ ]:
ID=re.compile(r'\b(?:RW-[0-9A-F]{10}|NOVA-[0-9A-F]{8})\b',re.I)
PAT=[(p,re.compile(r'(?<!\w)'+re.escape(a)+r'(?!\w)',re.I)) for p,ls in A.items() for a in sorted(set(ls[0]+ls[1]))]
def concepts(t):
    hits=[(m.start(),m.end(),p) for p,pat in PAT for m in pat.finditer(t)];keep=[]
    for a,b,p in sorted(hits,key=lambda x:(-(x[1]-x[0]),x[0],x[2])):
        if not any(a<y and b>x for x,y,_ in keep):keep.append((a,b,p))
    return sorted(keep)
def ids(t):return {m.group().upper() for m in ID.finditer(t)}
VP={p:re.compile(v,re.I) for p,v in {'budget':r'\bMT\s+\d+\b','deadline':r'\b20\d{2}-\d{2}-\d{2}\b','capacity':r'\b\d+\s+units\b','location':r'\bZONE-[A-H]{3}\b','status':r'\b(?:APPROVED|IN_REVIEW|PENDING|SUSPENDED)\b','duration':r'\b\d+\s+days\b','temperature':r'\b\d+\s*°C\b','pressure':r'\b\d+\s+kPa\b'}.items()}
NEG=re.compile(r'\b(?:não|not|rejeitad\w*|rejected|withdrawn)\b',re.I);UNC=re.compile(r'\b(?:proposta|proposed|poderá|may be|por validar|awaiting validation)\b',re.I);LINK=re.compile(r'(?:=|:|\b(?:é|is|of|em|at|as|de|totals)\b)',re.I)
def parse(d):
    out=[]
    for m in re.finditer(r'[^;.\n]+',d['text']):
        raw=m.group();t=raw.strip();a=m.start()+len(raw)-len(raw.lstrip());cs=concepts(t)
        out.append((a,a+len(t),t,ids(t),cs,{p:list(VP[p].finditer(t)) for _,_,p in cs},bool(NEG.search(t)),bool(UNC.search(t))))
    return out
P={s:parse(d) for s,d in D.items()};QC={s:(ids(q['text']),{p for _,_,p in concepts(q['text'])}) for s,q in Q.items()}
def extract(qid,sid):
    qi,qp=QC[qid];stats=[0]*6;sp=[]
    for a,b,t,di,cs,vs,neg,unc in P[sid]:
        cp={p for _,_,p in cs}
        if not(qi&di and qp&cp):continue
        stats[0]+=1;stats[1]+=int(neg);stats[2]+=int(unc)
        for p in sorted(qp&cp):
            vv=vs[p];stats[3]+=bool(vv)
            for v in vv:
                link=any(e<=v.start() and LINK.search(t[e:v.start()]) for _,e,x in cs if x==p);stats[4]+=int(link)
                if len(qi)==1 and len(qp)==1 and cp==qp and len(vv)==1 and link and not neg and not unc and all(sum(i.startswith(pr) for i in di)<=1 for pr in ('RW-','NOVA-')):sp.append({'source_id':sid,'property':p,'value':v.group(),'start':a,'end':b,'quote':t})
    stats[5]=len(sp);return sp,stats
def scoped(q,d):return q['split']==d['split'] and q['scope']==d['scope']
def eligible(q,d):return scoped(q,d) and d['authority']=='OFFICIAL_SOURCE' and d['observed_at']<=q['as_of'] and d['valid_from']<=q['as_of'] and (d['valid_until'] is None or q['as_of']<d['valid_until'])
assert all(eligible(Q[qid],D[a['source_id']]) for qid,g in gold.items() for a in g['support'])
QC['probe']=(ids('due date RW-ABCDEF0123'),{'deadline'})
probes=[('RW-ABCDEF0123 due date: 2026-08-20',1),('RW-ABCDEF0123 capacity: 25 units; due date noted',0),('RW-ABCDEF0123 due date: 2026-08-20 was rejected',0),('Proposed due date of RW-ABCDEF0123: 2026-08-20',0),('RW-1111111111 due date: 2026-08-20',0),('RW-ABCDEF0123 limit: 25 units',0)]
for text,want in probes:
    P['probe']=parse({'text':text});assert len(extract('probe','probe')[0])==want
P.pop('probe');QC.pop('probe');print('Parser probes:',len(probes),'PASS')
train=[d for d in docs if d['split']=='train' and d['scope']!='lab:foreign']
def tok(t):return re.findall(r'[^\W_]+',t.casefold())
DT={s:Counter(tok(d['text'])) for s,d in D.items()};df=Counter()
for d in train:df.update(DT[d['source_id']].keys())
N=len(train);avg=sum(sum(DT[d['source_id']].values()) for d in train)/N;idf={t:math.log(1+(N-n+.5)/(n+.5)) for t,n in df.items()};unseen=math.log(1+(N+.5)/.5)
def bm(q,s):
    c=DT[s];norm=1.2*(.25+.75*sum(c.values())/avg);return sum(idf.get(t,unseen)*c[t]*2.2/(c[t]+norm) for t in sorted(set(tok(q['text']))) if c[t])
vec=TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,sublinear_tf=True);vec.fit([d['text'] for d in train]);DX=vec.transform([d['text'] for d in docs]);QX=vec.transform([q['text'] for q in queries]);DI={d['source_id']:i for i,d in enumerate(docs)};QI={q['query_id']:i for i,q in enumerate(queries)}
FEATURES=['bm25_relative','char_similarity','token_overlap','entity_match','property_mention','mention_joint','clause_match','negated','uncertain','typed_value','connector','strict_span']
F={};POOL={};SP={};SCOPE={};DC={s:{p for _,_,p in concepts(d['text'])} for s,d in D.items()};DID={s:ids(d['text']) for s,d in D.items()}
for q in queries:
    qid=q['query_id'];pool=[s for s,d in D.items() if eligible(q,d)];scope=[s for s,d in D.items() if scoped(q,d)];SCOPE[qid]=sorted([(s,bm(q,s)) for s in scope],key=lambda a:(-a[1],a[0]));b=np.array([bm(q,s) for s in pool]);b=b/max(float(b.max()),1e-12);ch=(DX[[DI[s] for s in pool]]@QX[QI[qid]].T).toarray().ravel();qt=set(tok(q['text']));qi,qp=QC[qid];f=[];sp={}
    for i,s in enumerate(pool):
        ident=float(bool(qi&DID[s]));prop=float(bool(qp) and qp.issubset(DC[s]));overlap=len(qt&set(DT[s]))/max(1,len(qt));sp[s],st=extract(qid,s);f.append([b[i],ch[i],overlap,ident,prop,ident*prop,*st])
    F[qid]=np.asarray(f);POOL[qid]=pool;SP[qid]=sp
assert all(np.isfinite(x).all() for x in F.values());print('Eligible pairs',sum(map(len,POOL.values())))

### 3. Fit models
Pointwise and pairwise use six lexical/mention features; learned support verifier adds six clause features. Each training query contributes equal total weight; positive and negative classes balanced per query where both exist. Pairwise uses six hard and six random negatives per support, symmetric differences. No test labels in fit.

In [ ]:
def explicit(x):return .45*x[:,0]+.25*x[:,1]+.3*x[:,2]+1.25*x[:,3]+.8*x[:,5]
def hybrid(x):return .5*x[:,0]+.3*x[:,1]+.2*x[:,2]
X=[];Y=[];W=[];PX=[];PY=[];PW=[];rows=[];pairs=[];prng=random.Random(SEED+1)
for qid in sorted(s for s,q in Q.items() if q['split']=='train'):
    x=F[qid];pool=POOL[qid];truth={a['source_id'] for a in gold[qid]['support']};y=np.array([int(s in truth) for s in pool]);pos=np.flatnonzero(y);neg=np.flatnonzero(1-y);w=np.full(len(y),1/len(y)) if not len(pos) else np.where(y==1,.5/len(pos),.5/len(neg));X.extend(x);Y.extend(y);W.extend(w)
    rows.extend({'query_id':qid,'source_id':s,'label':int(yi),'weight':float(wi)} for s,yi,wi in zip(pool,y,w));es=explicit(x)
    for pi in pos:
        hard=sorted(neg,key=lambda i:(-es[i],pool[i]))[:6];rand=prng.sample([i for i in neg if i not in hard],6)
        for ni in hard+rand:
            d=x[pi,:6]-x[ni,:6];PX.extend([d,-d]);PY.extend([1,0]);PW.extend([1/(24*len(pos))]*2);pairs.append({'query_id':qid,'positive_source':pool[pi],'negative_source':pool[ni],'selection':'hard' if ni in hard else 'random'})
X=np.asarray(X);Y=np.asarray(Y);W=np.asarray(W);PX=np.asarray(PX);PY=np.asarray(PY);PW=np.asarray(PW)
def fit(x,y,w):
    sc=StandardScaler().fit(x);m=LogisticRegression(C=1,solver='liblinear',random_state=SEED,max_iter=1000).fit(sc.transform(x),y,sample_weight=w/w.mean());return sc,m
point=fit(X[:,:6],Y,W);pair=fit(PX,PY,PW);ver=fit(X,Y,W)
def predict(model,x,prob=False):
    sc,m=model;z=sc.transform(x);return m.predict_proba(z)[:,1] if prob else m.decision_function(z)
def params(model,f):
    sc,m=model;return {'features':f,'mean':sc.mean_.tolist(),'scale':sc.scale_.tolist(),'weights':m.coef_[0].tolist(),'intercept':float(m.intercept_[0])}
write('weights.json',{'pointwise':params(point,FEATURES[:6]),'pairwise':params(pair,FEATURES[:6]),'verifier':params(ver,FEATURES),'note':'Experimental coefficients; not packaged inference pipeline'})
lines('training_rows.jsonl',rows);lines('training_pairs.jsonl',pairs);print('Training rows',len(rows),'positives',int(Y.sum()),'pairwise',len(pairs),'directions',len(PY))

## Results
### 4. Retrieval, support and decision
Seven retrieval methods including scope-only control. Three policies use the same explicit top five. MentionTop1 can propose a source without a valid span (penalized). AssertionRule and LearnedVerifier require one strict span; conflicting values from different parsed sources trigger conflict. Ambiguous query abstains. Dev thresholds are frozen before diagnostic predictions.

In [ ]:
METHODS=['BM25-Scope','BM25','Hybrid','Explicit','Pointwise','Pairwise','SupportVerifier'];POLICIES=['MentionTop1','AssertionRule','LearnedVerifier'];ranks=[];dec=[]
def rank(qid,v):return sorted(range(len(v)),key=lambda i:(-float(v[i]),POOL[qid][i]))[:K]
def build(split):
 for qid,q in Q.items():
    if q['split']!=split:continue
    x=F[qid];pool=POOL[qid];pred=predict(ver,x,True);scores={'BM25':x[:,0],'Hybrid':hybrid(x),'Explicit':explicit(x),'Pointwise':predict(point,x[:,:6]),'Pairwise':predict(pair,x[:,:6]),'SupportVerifier':pred};sr=SCOPE[qid][:K]
    ranks.append({'query_id':qid,'method':'BM25-Scope','candidate_count':len(SCOPE[qid]),'source_ids':[s for s,v in sr],'scores':[v for s,v in sr]})
    for name,v in scores.items():
        rr=rank(qid,v);ranks.append({'query_id':qid,'method':name,'candidate_count':len(pool),'source_ids':[pool[i] for i in rr],'scores':[float(v[i]) for i in rr]})
    short=rank(qid,scores['Explicit']);top=short[0];shortids=[pool[i] for i in short];ev=[(i,a) for i in short for a in SP[qid][pool[i]]];conflict=None
    for i,a in ev:
        for j,b in ev:
            if i!=j and a['value'].casefold()!=b['value'].casefold():conflict=[a,b];break
        if conflict:break
    qi,qp=QC[qid];mspan=SP[qid][pool[top]];mc=float(x[top,5]*(.5+.5*np.tanh(max(0,scores['Explicit'][top]-scores['Explicit'][short[1]]))))
    for pol in POLICIES:
        rec={'query_id':qid,'policy':pol,'shortlist':shortids,'proposed_source':None,'evidence':None,'conflict_evidence':None,'confidence':0.,'fixed_action':None,'parsed_candidates':[a for i,a in ev]}
        if len(qi)!=1 or len(qp)!=1:rec['fixed_action']='ABSTAIN'
        elif pol=='MentionTop1':rec.update(proposed_source=pool[top],evidence=mspan[0] if len(mspan)==1 else None,confidence=mc)
        elif conflict:rec.update(fixed_action='CONFLICT',conflict_evidence=conflict)
        else:
            clean=[(i,a) for i,a in ev if len(SP[qid][pool[i]])==1]
            if clean:
                i,a=clean[0] if pol=='AssertionRule' else sorted(clean,key=lambda z:(-pred[z[0]],short.index(z[0])))[0];rec.update(proposed_source=pool[i],evidence=a,confidence=1. if pol=='AssertionRule' else float(pred[i]))
            else:rec['fixed_action']='ABSTAIN'
        dec.append(rec)
def match(a,g):return a is not None and any(a==s for s in g['support'])
def action(r,t):return r['fixed_action'] or ('EVIDENCE_CANDIDATE' if r['proposed_source'] and r['confidence']>=t and r['confidence']>0 else 'ABSTAIN')
def assess(rr,t):
    c=Counter();n=len(rr)
    for r in rr:
        g=gold[r['query_id']];a=action(r,t);positive=g['class'].startswith('ANSWERABLE');c[a]+=1
        if a=='EVIDENCE_CANDIDATE':
            src=positive and r['proposed_source'] in {x['source_id'] for x in g['support']};ok=src and match(r['evidence'],g);c['source_correct']+=int(src);c['correct']+=int(ok);c['wrong']+=int(not ok);c['wrong_on_answerable']+=int(positive and not ok);c['negative_false_accept']+=int(not positive)
        elif a=='CONFLICT':
            e=r['conflict_evidence'];ok=g['class']=='CONFLICT' and len(e)==2 and all(match(x,g) for x in e) and {x['source_id'] for x in e}=={x['source_id'] for x in g['support']};c['conflict_exact']+=int(ok);c['conflict_wrong']+=int(not ok)
    accepted=c['EVIDENCE_CANDIDATE'];return {'n':n,'threshold':t,'accepted':accepted,**{k:c[k] for k in ['correct','wrong','source_correct','wrong_on_answerable','negative_false_accept','conflict_exact','conflict_wrong']},'conflict_signals':c['CONFLICT'],'abstain':c['ABSTAIN'],'coverage':accepted/n if n else None,'risk':c['wrong']/accepted if accepted else None}
build('train');build('dev');curve=[];thresholds={}
for pol in POLICIES:
    rr=[r for r in dec if Q[r['query_id']]['split']=='dev' and r['policy']==pol];outcomes=[dict(policy=pol,**assess(rr,t)) for t in CFG['grid']];curve.extend(outcomes);thresholds[pol]=max([r for r in outcomes if r['wrong']==0],key=lambda r:(r['correct'],r['threshold']))['threshold']
write('frozen_thresholds.json',thresholds);lines('dev_threshold_grid.jsonl',curve);print('Frozen thresholds',thresholds)
build('diagnostic')
assert len(ranks)==len(Q)*len(METHODS) and len(dec)==len(Q)*len(POLICIES)
assert len({(r['query_id'],r['method']) for r in ranks})==len(ranks)
assert all(len(r['source_ids'])==len(set(r['source_ids'])) and all(scoped(Q[r['query_id']],D[s]) and (r['method']=='BM25-Scope' or eligible(Q[r['query_id']],D[s])) for s in r['source_ids']) for r in ranks)
assert all(D[a['source_id']]['text'][a['start']:a['end']]==a['quote'] for r in dec for a in r['parsed_candidates'])
lines('rankings.jsonl',ranks);lines('decision_inputs.jsonl',dec)

### 5. Metrics and failure analysis
Binary multi-source nDCG, MRR and hit for support-bearing queries; separate ordinary positives and conflicts. Acceptance risk requires exact source/value/span. Conflicts evaluated separately. Group bootstrap is descriptive only.

In [ ]:
def rmetric(r):
    wanted={a['source_id'] for a in gold[r['query_id']]['support']};found=[i+1 for i,s in enumerate(r['source_ids']) if s in wanted];first=min(found,default=K+1);dcg=sum(1/math.log2(i+1) for i in found);ideal=sum(1/math.log2(i+1) for i in range(1,min(K,len(wanted))+1));return {'hit1':float(first==1),'hit5':float(first<=K),'mrr5':1/first if first<=K else 0.,'ndcg5':dcg/ideal}
metrics=[];dm=[];final=[]
for split in CFG['groups']:
 for method in METHODS:
    for cohort in ['ALL_SUPPORT','ANSWERABLE','CONFLICT','PT','EN']:
        rr=[r for r in ranks if r['method']==method and Q[r['query_id']]['split']==split and gold[r['query_id']]['support'] and (cohort=='ALL_SUPPORT' or cohort=='ANSWERABLE' and gold[r['query_id']]['class'].startswith('ANSWERABLE') or cohort=='CONFLICT' and gold[r['query_id']]['class']=='CONFLICT' or cohort.lower()==Q[r['query_id']]['language'])];ms=[rmetric(r) for r in rr];metrics.append({'split':split,'method':method,'cohort':cohort,'n':len(ms),**{k:float(np.mean([m[k] for m in ms])) if ms else None for k in ['hit1','hit5','mrr5','ndcg5']}})
 for pol in POLICIES:
    rr=[r for r in dec if Q[r['query_id']]['split']==split and r['policy']==pol];t=thresholds[pol]
    for cohort in ['ALL','ANSWERABLE','ANSWERABLE_ID','ANSWERABLE_ALIAS','MISSING','NEGATED','UNCERTAIN','FOREIGN_ONLY','EXPIRED_ONLY','CONFLICT','AMBIGUOUS']:
        sub=[r for r in rr if cohort=='ALL' or cohort=='ANSWERABLE' and gold[r['query_id']]['class'].startswith('ANSWERABLE') or gold[r['query_id']]['class']==cohort];dm.append({'split':split,'policy':pol,'cohort':cohort,**assess(sub,t)})
    for r in rr:final.append({'query_id':r['query_id'],'policy':pol,'threshold':t,'action':action(r,t),'proposed_source':r['proposed_source'],'evidence':r['evidence'],'conflict_evidence':r['conflict_evidence']})
by={(r['query_id'],r['method']):rmetric(r)['ndcg5'] for r in ranks if Q[r['query_id']]['split']=='diagnostic' and gold[r['query_id']]['support']};delta=defaultdict(list)
for qid in Q:
    if (qid,'Explicit') in by:delta[gold[qid]['group']].append(by[qid,'SupportVerifier']-by[qid,'Explicit'])
a=np.array([np.mean(delta[g]) for g in sorted(delta)]);boot=np.random.default_rng(SEED+2).choice(a,(2000,len(a)),replace=True).mean(axis=1);paired={'groups':len(a),'mean':float(a.mean()),'percentiles':np.percentile(boot,[2.5,97.5]).tolist(),'group_deltas':dict(zip(sorted(delta),a.tolist())),'descriptive_only':True}
lines('retrieval_metrics.jsonl',metrics);lines('decision_metrics.jsonl',dm);lines('decisions.jsonl',final);write('paired_effect.json',paired)
R=pd.DataFrame(metrics);M=pd.DataFrame(dm);rr=R[(R.split=='diagnostic')&(R.cohort=='ALL_SUPPORT')];dd=M[(M.split=='diagnostic')&(M.cohort=='ALL')]
display(rr.round(4));display(dd[['policy','threshold','n','accepted','correct','wrong','negative_false_accept','conflict_exact','conflict_wrong','abstain','coverage','risk']].round(4));print('Paired effect',paired['mean'],paired['percentiles'])
fail=[]
for r in final:
    if r['policy']!='LearnedVerifier' or Q[r['query_id']]['split']!='diagnostic':continue
    g=gold[r['query_id']];kind=g['class'];good=(kind.startswith('ANSWERABLE') and r['action']=='EVIDENCE_CANDIDATE' and match(r['evidence'],g)) or (kind=='CONFLICT' and r['action']=='CONFLICT' and all(match(x,g) for x in r['conflict_evidence']) and {x['source_id'] for x in r['conflict_evidence']}=={x['source_id'] for x in g['support']}) or (not g['support'] and r['action']=='ABSTAIN')
    if not good:fail.append({'query':Q[r['query_id']],'oracle':g,'decision':r})
lines('diagnostic_failures.jsonl',fail);print('Learned diagnostic failures',len(fail))

### 6. Visual diagnostics
Scope-only comparator differs in eligibility. Risk undefined if none accepted. Diagnostic outcomes shown without tuning.

In [ ]:
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(figsize=(9,4));b=ax.barh(rr.method,rr.ndcg5);ax.invert_yaxis();ax.set_xlim(0,1.12);ax.set(xlabel='nDCG@5 — 48 support queries',title='Diagnostic retrieval (scope-only control separate)');ax.bar_label(b,fmt='%.3f',padding=3);fig.tight_layout();fig.savefig(OUT/'retrieval.png');plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4))
for pol in POLICIES:
    pts=[r for r in curve if r['policy']==pol and r['risk'] is not None];ax.scatter([r['coverage'] for r in pts],[r['risk'] for r in pts],label=pol,s=18,alpha=.75)
ax.set(xlabel='Coverage / 144 dev queries',ylabel='Wrong / accepted',title='Dev-only risk–coverage');ax.legend();ax.grid(alpha=.2);fig.tight_layout();fig.savefig(OUT/'risk_coverage.png');plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4));bottom=np.zeros(len(dd))
for col in ['correct','wrong','conflict_exact','conflict_wrong','abstain']:
    v=dd[col].to_numpy();ax.barh(dd.policy,v,left=bottom,label=col);bottom+=v
assert np.all(bottom==dd.n.to_numpy());ax.set(xlabel='Queries (144)',title='Diagnostic decisions after threshold freeze');ax.legend(loc='upper center',bbox_to_anchor=(.5,-.15),ncol=3,fontsize=8);fig.tight_layout();fig.savefig(OUT/'decisions.png',bbox_inches='tight');plt.show();plt.close(fig)

## Takeaways and export
Results only support this synthetic experiment. No scientific gate changes. Download ZIP and return for independent audit. Do not retune on diagnostic. Independent human-reviewed holdout remains mandatory.

In [ ]:
summary={'experiment':'E1-X03','status':'EXECUTED_PROVISIONAL_NOT_INDEPENDENT','science_gate':'REVIEW_REQUIRED','human_reviews':0,'counts':{'documents':len(docs),'queries':len(queries),'groups':72,'training_rows':len(rows),'training_positive_rows':int(Y.sum()),'pairwise_comparisons':len(pairs),'pairwise_directions':len(PY),'rankings':len(ranks),'decision_inputs':len(dec),'retrieval_metric_rows':len(metrics),'decision_metric_rows':len(dm),'threshold_rows':len(curve),'parser_probes':len(probes)},'thresholds':thresholds,'diagnostic_retrieval':[r for r in metrics if r['split']=='diagnostic' and r['cohort']=='ALL_SUPPORT'],'diagnostic_decisions':[r for r in dm if r['split']=='diagnostic' and r['cohort']=='ALL'],'scope_violations_observed':0,'environment':{'python':sys.version.split()[0],'numpy':np.__version__,'sklearn':sklearn.__version__,'platform':platform.platform(),'utc':datetime.now(timezone.utc).isoformat()}}
write('summary.json',summary);manifest={'schema':'RIGHTWARE_E1_X03_MANIFEST_V1','entries':[{'path':p.name,'size':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(OUT.iterdir()) if p.is_file()]};write('manifest.json',manifest)
tmp=ZIP.with_name(ZIP.name+'.'+uuid.uuid4().hex+'.tmp')
with zipfile.ZipFile(tmp,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):z.write(p,p.name)
with zipfile.ZipFile(tmp) as z:
    assert z.testzip() is None and set(z.namelist())=={x['path'] for x in manifest['entries']}|{'manifest.json'}
    for e in manifest['entries']:
        b=z.read(e['path']);assert len(b)==e['size'] and hashlib.sha256(b).hexdigest()==e['sha256']
os.replace(tmp,ZIP);print('E1_X03_EXECUTION_COMPLETE',ZIP,'files',len(manifest['entries']),'SHA256',hashlib.sha256(ZIP.read_bytes()).hexdigest());print('E1_DATA_READY=REVIEW_REQUIRED')